# DKVMN Next-Item — Colab

Notebook pronto para testar **DKVMN Next-Item** no Google Colab, com:

- setup do ambiente
- suporte a repositório privado via **ZIP no Google Drive**
- opção de usar **sequências por sessão**
- treino inline com **loss mascarada** e **AUC**
- gráficos de **Train/Val Loss** e **Train/Val AUC**


In [ ]:
from pathlib import Path
import os
import sys
import shutil
import zipfile

IN_COLAB = "google.colab" in sys.modules
REPO_DIR = Path("/content/ai-core")

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    ZIP_PATH = Path("/content/drive/MyDrive/ai-core.zip")

    if REPO_DIR.exists():
        shutil.rmtree(REPO_DIR)

    if ZIP_PATH.exists():
        with zipfile.ZipFile(ZIP_PATH, "r") as zf:
            zf.extractall("/content")
        print(f"Repo extraído de {ZIP_PATH}")
    else:
        raise FileNotFoundError("Coloque ai-core.zip em /content/drive/MyDrive/ai-core.zip")

    os.chdir(REPO_DIR)
    get_ipython().system('pip install -q -r requirements.txt')
else:
    REPO_DIR = Path.cwd()

SRC_PATH = REPO_DIR / "src"
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

print("Repo:", REPO_DIR)
print("SRC_PATH:", SRC_PATH)


In [ ]:
from pathlib import Path
import shutil

PROJECT_ROOT = REPO_DIR
DATA_TARGET = PROJECT_ROOT / "data"
DATA_TARGET.mkdir(parents=True, exist_ok=True)

SOURCE_RAW_DIR = Path("/content/drive/MyDrive/data/raw")
SOURCE_PROCESSED_DIR = Path("/content/drive/MyDrive/data/processed")

if SOURCE_RAW_DIR.exists():
    shutil.copytree(SOURCE_RAW_DIR, DATA_TARGET / "raw", dirs_exist_ok=True)
if SOURCE_PROCESSED_DIR.exists():
    shutil.copytree(SOURCE_PROCESSED_DIR, DATA_TARGET / "processed", dirs_exist_ok=True)

checks = [
    Path("data/raw/answers.json"),
    Path("data/processed/dataset/answers_prepared.csv"),
    Path("data/processed/sequences/user_sequences.json"),
]
for p in checks:
    print(f"{p}: {p.exists()}")


In [ ]:
# Rode se precisar gerar os artefatos base.
# get_ipython().system('python run_data_pipeline.py')
# Opcional e recomendado: reconstruir as sequências por sessão.
# get_ipython().system('python scripts/build_user_session_sequences.py')
print("Descomente as linhas acima se precisar regenerar os artefatos.")


In [ ]:
import json
import math
import random
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from sklearn.metrics import roc_auc_score

from brain_kt.dataset.kt_next_item_dataset import KTNextItemDataset, build_id_mappings
from brain_kt.models.dkvmn_next_item import DKVMNNextItemModel
from brain_kt.preprocessing.build_next_item_training_sequences import build_next_item_training_sequences

def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(42)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)


In [ ]:
SEQUENCES_PATH = Path("data/processed/sequences/user_sequences.json")
with open(SEQUENCES_PATH, "r", encoding="utf-8") as f:
    sequences = json.load(f)

training_data = build_next_item_training_sequences(
    sequences,
    max_seq_len=50,
    stride=25,
    min_seq_len=3,
)

print("User/session sequences:", len(sequences))
print("Training windows:", len(training_data))


In [ ]:
user_ids = list({item["user_id"] for item in training_data})
random.shuffle(user_ids)
n_users = len(user_ids)

train_users = set(user_ids[: int(0.7 * n_users)])
val_users = set(user_ids[int(0.7 * n_users): int(0.85 * n_users)])
test_users = set(user_ids[int(0.85 * n_users):])

train = [item for item in training_data if item["user_id"] in train_users]
val = [item for item in training_data if item["user_id"] in val_users]
test = [item for item in training_data if item["user_id"] in test_users]

maps = build_id_mappings(train)

train_ds = KTNextItemDataset(train, maps.question_to_idx, maps.skill_to_idx, max_seq_len=50)
val_ds = KTNextItemDataset(val, maps.question_to_idx, maps.skill_to_idx, max_seq_len=50)
test_ds = KTNextItemDataset(test, maps.question_to_idx, maps.skill_to_idx, max_seq_len=50)

train_loader = DataLoader(train_ds, batch_size=16, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=16)
test_loader = DataLoader(test_ds, batch_size=16)

print("Train windows:", len(train_ds))
print("Val windows:", len(val_ds))
print("Test windows:", len(test_ds))


In [ ]:
model = DKVMNNextItemModel(
    num_questions=len(maps.question_to_idx),
    num_skills=len(maps.skill_to_idx),
    memory_size=20,
    key_dim=32,
    value_dim=64,
    dropout=0.2,
).to(DEVICE)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-5)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", factor=0.5, patience=2)

all_targets = [c for item in train for c in item["target"]["next_corrects"]]
n_pos = sum(all_targets)
n_neg = len(all_targets) - n_pos
pos_weight = torch.tensor([n_neg / max(n_pos, 1)], device=DEVICE)
criterion = nn.BCEWithLogitsLoss(reduction="none", pos_weight=pos_weight)

sum(p.numel() for p in model.parameters())


In [ ]:
def compute_auc_from_tensors(probs, targets):
    probs_np = probs.detach().cpu().numpy()
    targets_np = targets.detach().cpu().numpy()
    if len(np.unique(targets_np)) < 2:
        return 0.5
    return float(roc_auc_score(targets_np, probs_np))

def evaluate(model, loader, criterion):
    model.eval()
    total_loss = 0.0
    total_weight = 0.0
    all_probs, all_targets = [], []

    with torch.no_grad():
        for batch in loader:
            batch = {k: v.to(DEVICE) for k, v in batch.items()}
            logits = model(batch)
            raw_loss = criterion(logits, batch["targets"])
            mask = batch["mask"]
            batch_loss = (raw_loss * mask).sum() / mask.sum()

            weight = mask.sum().item()
            total_loss += batch_loss.item() * weight
            total_weight += weight

            probs = torch.sigmoid(logits)
            all_probs.append(probs[mask])
            all_targets.append(batch["targets"][mask])

    probs = torch.cat(all_probs)
    targets = torch.cat(all_targets)
    auc = compute_auc_from_tensors(probs, targets)
    acc = ((probs > 0.5) == targets).float().mean().item()
    avg_loss = total_loss / max(total_weight, 1.0)
    return avg_loss, auc, acc


In [ ]:
EPOCHS = 10
PATIENCE = 4
history = []
best_state = None
best_val_auc = 0.0
best_val_loss = float("inf")
no_improve = 0

for epoch in range(EPOCHS):
    model.train()
    train_loss_sum = 0.0
    train_weight = 0.0
    train_probs, train_targets = [], []

    for step, batch in enumerate(train_loader, start=1):
        batch = {k: v.to(DEVICE) for k, v in batch.items()}
        logits = model(batch)
        raw_loss = criterion(logits, batch["targets"])
        mask = batch["mask"]
        loss = (raw_loss * mask).sum() / mask.sum()

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        weight = mask.sum().item()
        train_loss_sum += loss.item() * weight
        train_weight += weight

        probs = torch.sigmoid(logits)
        train_probs.append(probs[mask].detach())
        train_targets.append(batch["targets"][mask].detach())

        if step % 50 == 0:
            print(f"Epoch {epoch+1} | Step {step}/{len(train_loader)}")

    train_probs = torch.cat(train_probs)
    train_targets = torch.cat(train_targets)
    train_loss = train_loss_sum / max(train_weight, 1.0)
    train_auc = compute_auc_from_tensors(train_probs, train_targets)
    train_acc = ((train_probs > 0.5) == train_targets).float().mean().item()

    val_loss, val_auc, val_acc = evaluate(model, val_loader, criterion)
    scheduler.step(val_auc)

    row = {
        "epoch": epoch + 1,
        "train_loss": train_loss,
        "train_auc": train_auc,
        "train_acc": train_acc,
        "val_loss": val_loss,
        "val_auc": val_auc,
        "val_acc": val_acc,
        "lr": optimizer.param_groups[0]["lr"],
    }
    history.append(row)

    print(
        f"Epoch {epoch+1}/{EPOCHS} | Train Loss: {train_loss:.4f} | Train AUC: {train_auc:.4f} | "
        f"Val Loss: {val_loss:.4f} | Val AUC: {val_auc:.4f} | Val Acc: {val_acc:.4f} | "
        f"LR: {optimizer.param_groups[0]['lr']:.2e}"
    )

    if val_auc > best_val_auc:
        best_val_auc = val_auc
        best_val_loss = val_loss
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        no_improve = 0
    else:
        no_improve += 1
        if no_improve >= PATIENCE:
            print(f"Early stopping na epoch {epoch+1}")
            break

history_df = pd.DataFrame(history)
history_df


In [ ]:
if best_state is not None:
    model.load_state_dict(best_state)

test_loss, test_auc, test_acc = evaluate(model, test_loader, criterion)
print(f"Best Val AUC: {best_val_auc:.4f}")
print(f"Best Val Loss: {best_val_loss:.4f}")
print(f"Test Loss: {test_loss:.4f} | Test AUC: {test_auc:.4f} | Test Acc: {test_acc:.4f}")


In [ ]:
fig = plt.figure(figsize=(10, 5))
plt.plot(history_df["epoch"], history_df["train_loss"], label="Train Loss")
plt.plot(history_df["epoch"], history_df["val_loss"], label="Val Loss")
plt.title("DKVMN Next-Item — Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.grid(alpha=0.3)
plt.show()


In [ ]:
fig = plt.figure(figsize=(10, 5))
plt.plot(history_df["epoch"], history_df["train_auc"], label="Train AUC")
plt.plot(history_df["epoch"], history_df["val_auc"], label="Val AUC")
plt.title("DKVMN Next-Item — AUC")
plt.xlabel("Epoch")
plt.ylabel("AUC")
plt.legend()
plt.grid(alpha=0.3)
plt.show()


In [ ]:
save_dir = Path("artifacts/colab_dkvmn")
save_dir.mkdir(parents=True, exist_ok=True)

history_df.to_csv(save_dir / "dkvmn_history.csv", index=False)
torch.save(model.state_dict(), save_dir / "dkvmn_next_item_colab.pt")

summary = {
    "best_val_auc": float(best_val_auc),
    "best_val_loss": float(best_val_loss),
    "test_loss": float(test_loss),
    "test_auc": float(test_auc),
    "test_acc": float(test_acc),
}

with open(save_dir / "metrics.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

summary
